## Query-Ürün Alaka Tahmini Pipeline'ı

**Görev:** Verilen bir (query, ürün) çifti için ürünün arama terimiyle alakalı (1) veya alakasız (0) olduğunu tahmin etmek. Değerlendirme metriği: Macro F1.

---

**Yaklaşım:**

EDA analizinde iki kritik bulgu tespit edildi:

1. Pozitif çiftlerin **%24'ünde** query ile ürün başlığı arasında hiç kelime örtüşmesi yoktur. "sneaker" → "koşu ayakkabısı" gibi semantik bağlantılar klasik kelime eşleştirme yöntemleriyle yakalanamaz.

2. Pozitif çiftlerin **%28'inde** query kelimeleri ürün başlığında birebir geçmektedir. Bu çiftler için kelime eşleşmesi güçlü bir sinyaldir.

Bu iki örüntüyü birlikte yakalamak amacıyla BERT ve LightGBM modellerinin olasılıkları ensemble yöntemiyle birleştirilmiştir.

---

**Aşama 1 — BERT Fine-Tuning:**
- Model: dbmdz/bert-base-turkish-cased (110M parametre, Türkçe pre-trained)
- Veri: 500K karma dataset (250K pozitif + 125K rastgele negatif + 125K E5 hard negative)
- Epoch: 3 | Learning rate: 1e-5 | Batch size: 32 | Max token: 64
- Weight decay: 0.01 | Gradient clipping: 1.0 | Linear LR scheduler

---

**Aşama 2 — LightGBM Ensemble:**
- Featurelar: TF-IDF skoru, title/category/brand benzerliği, Jaccard, recall, query uzunluğu
- BERT olasılıkları (%80) + LightGBM olasılıkları (%20) ağırlıklı ortalama
- Threshold optimizasyonu: threshold = 0.35 → Kaggle Public LB: 0.86

## Kütüphaneler ve Veri Yükleme

**Yüklenen dosyalar:**
- dataset.csv → 500K eğitim verisi (preprocessing notebook çıktısı)
- items_clean.csv → temizlenmiş 962K ürün kataloğu
- terms_clean.csv → temizlenmiş 50K arama terimi
- submission_pairs.csv → tahmin edilecek 3.3M çift

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

PATH = "C:/Users/acely/OneDrive/Masaüstü/trendyol_final/"

dataset = pd.read_csv(PATH + "dataset.csv")
items_clean = pd.read_csv(PATH + "items_clean.csv")
terms_clean = pd.read_csv(PATH + "terms_clean.csv")
sub = pd.read_csv(PATH + "submission_pairs.csv")

print(f"✅ Veriler yüklendi")
print(f"Dataset: {len(dataset):,} | Sub: {len(sub):,}")

Cihaz: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
✅ Veriler yüklendi
Dataset: 500,000 | Sub: 3,359,679


## Tokenizer ve Dataset Sınıfı

**Tokenizer :**
BERT metni doğrudan okuyamaz, önce kelimeleri sayılara dönüştürmek gerekir. Tokenizer bu işlemi yapar.

**Dataset sınıfı :**
500K veriyi aynı anda belleğe alamayız. Dataset sınıfı model "bana N. örneği ver" dediğinde o örneği tokenize edip hazır hale getirir. DataLoader ise bunları 32'lik gruplar (batch) halinde modele iletir.

**max_length=64:**
Her çift için maksimum 64 token okunur. Query'ler ortalama 2.6 kelime, item_text kısa tutulduğu için 64 token yeterlidir.

In [2]:
MODEL_NAME = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SearchDataset(Dataset):
    def __init__(self, queries, item_texts, labels=None, tokenizer=None, max_length=64):
        self.queries = queries
        self.item_texts = item_texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.queries[idx], self.item_texts[idx],
            truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("✅ Tokenizer ve Dataset hazır")

✅ Tokenizer ve Dataset hazır


## BERT Model Eğitimi

Eğitim üç aşamada gerçekleştirilmiştir:

**Aşama 1 — İlk Eğitim (dataset_orig):**
- 375K veri: 250K pozitif + 125K orijinal negatif
- 3 epoch, lr=2e-5
- Çıktı: best_model_v2.pt

**Aşama 2 — Hard Negative Mining:**
- Aşama 1 modeli kendi negatiflerine uygulandı
- Model yanlışlıkla 1 dediği negatifler bulundu
- Bunlar dataset'e eklendi

**Aşama 3 — Mixed Dataset ile Fine-Tuning:**
- 500K karma veri: 250K pozitif + 125K orijinal negatif + 125K E5 hard negative
- 3 epoch, lr=1e-5
- Çıktı: best_model.pt (Kaggle Public LB: 0.86)

**Ortak Parametreler:**
- Batch boyutu: 32 | Max token: 64
- Weight decay: 0.01 | Gradient clipping: 1.0
- Linear LR scheduler (warmup %10)
- Her epoch sonunda Macro F1 ölçüldü, en iyi model kaydedildi

In [3]:
# Veri yükle
dataset_orig = pd.read_csv(PATH + "dataset_orig.csv")
dataset_mixed = pd.read_csv(PATH + "dataset.csv")

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return f1_score(all_preds, all_labels, average="macro")

def get_probs(model, loader, device):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
    return np.array(all_probs)

# === AŞAMA 1: İLK EĞİTİM (dataset_orig ile) ===
print("=== AŞAMA 1: İlk Eğitim ===")

train_df1, val_df1 = train_test_split(dataset_orig, test_size=0.1, random_state=42)
train_df1 = train_df1.reset_index(drop=True)
val_df1 = val_df1.reset_index(drop=True)

train_dataset1 = SearchDataset(train_df1["query"].tolist(), train_df1["item_text"].tolist(), train_df1["label"].tolist(), tokenizer)
val_dataset1 = SearchDataset(val_df1["query"].tolist(), val_df1["item_text"].tolist(), val_df1["label"].tolist(), tokenizer)
train_loader1 = DataLoader(train_dataset1, batch_size=32, shuffle=True, num_workers=0)
val_loader1 = DataLoader(val_dataset1, batch_size=64, shuffle=False, num_workers=0)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

EPOCHS = 3
optimizer1 = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps1 = len(train_loader1) * EPOCHS
scheduler1 = get_linear_schedule_with_warmup(optimizer1, num_warmup_steps=total_steps1//10, num_training_steps=total_steps1)

best_f1_v1 = 0
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader1, optimizer1, scheduler1, device)
    val_f1 = eval_epoch(model, val_loader1, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")
    if val_f1 > best_f1_v1:
        best_f1_v1 = val_f1
        torch.save(model.state_dict(), PATH + "best_model_v2.pt")
        print(f"✅ v2 kaydedildi! (F1: {best_f1_v1:.4f})")

print("✅ Aşama 1 tamamlandı!")

# === AŞAMA 2: HARD NEGATIVE MINING ===
print("\n=== AŞAMA 2: Hard Negative Mining ===")

model.load_state_dict(torch.load(PATH + "best_model_v2.pt"))

neg_df = dataset_orig[dataset_orig["label"] == 0].reset_index(drop=True)
neg_ds = SearchDataset(neg_df["query"].tolist(), neg_df["item_text"].tolist(), tokenizer=tokenizer)
neg_ld = DataLoader(neg_ds, batch_size=128, shuffle=False, num_workers=0)
neg_probs = get_probs(model, neg_ld, device)

hard_neg_mask = neg_probs >= 0.5
hard_neg_df = neg_df[hard_neg_mask].reset_index(drop=True)
print(f"✅ Hard negative sayısı: {len(hard_neg_df):,}")

dataset_v2 = pd.concat([
    dataset_orig[["query","item_text","label"]],
    hard_neg_df[["query","item_text","label"]]
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"✅ v2 dataset: {len(dataset_v2):,}")

# === AŞAMA 3: MIXED DATASET İLE FİNE-TUNING ===
print("\n=== AŞAMA 3: Mixed Dataset ile Fine-Tuning ===")

train_df2, val_df2 = train_test_split(dataset_mixed, test_size=0.1, random_state=42)
train_df2 = train_df2.reset_index(drop=True)
val_df2 = val_df2.reset_index(drop=True)

train_dataset2 = SearchDataset(train_df2["query"].tolist(), train_df2["item_text"].tolist(), train_df2["label"].tolist(), tokenizer)
val_dataset2 = SearchDataset(val_df2["query"].tolist(), val_df2["item_text"].tolist(), val_df2["label"].tolist(), tokenizer)
train_loader2 = DataLoader(train_dataset2, batch_size=32, shuffle=True, num_workers=0)
val_loader2 = DataLoader(val_dataset2, batch_size=64, shuffle=False, num_workers=0)

optimizer2 = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
total_steps2 = len(train_loader2) * 3
scheduler2 = get_linear_schedule_with_warmup(optimizer2, num_warmup_steps=total_steps2//10, num_training_steps=total_steps2)

best_f1 = 0
for epoch in range(3):
    train_loss = train_epoch(model, train_loader2, optimizer2, scheduler2, device)
    val_f1 = eval_epoch(model, val_loader2, device)
    print(f"Epoch {epoch+1}/3 | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), PATH + "best_model.pt")
        print(f"✅ Model kaydedildi! (F1: {best_f1:.4f})")

print("✅ Tüm eğitim tamamlandı!")

=== AŞAMA 1: İlk Eğitim ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Validation

Epoch 1/3 | Loss: 0.1011 | Val F1: 0.9806
✅ v2 kaydedildi! (F1: 0.9806)


Validation: 100%|██████████| 586/586 [01:10<00:00,  8.30it/s]


...

Epoch 1/3 | Loss: 0.3359 | Val F1: 0.8629
✅ Model kaydedildi! (F1: 0.8629)


Validation: 100%|██████████| 782/782 [01:33<00:00,  8.32it/s]


Epoch 2/3 | Loss: 0.2636 | Val F1: 0.8768
✅ Model kaydedildi! (F1: 0.8768)


Validation: 100%|██████████| 782/782 [01:33<00:00,  8.35it/s]


Epoch 3/3 | Loss: 0.2337 | Val F1: 0.8808
✅ Model kaydedildi! (F1: 0.8808)
✅ Tüm eğitim tamamlandı!


## Submission Olasılıklarının Üretilmesi

**Amaç:** Final submission için BERT ve LightGBM modellerinin olasılık skorlarını üretmek.

**Neden olasılık, 0/1 değil:**
Ensemble yaparken iki modelin kararını birleştirmek için 0-1 arası olasılık skoru gereklidir. 
Örneğin BERT 0.82, LightGBM 0.65 verirse → ağırlıklı ortalama ile 0.787 hesaplanır → threshold ile final karar verilir.

**BERT submission:**
- En iyi model (best_model.pt) yüklenir
- fp16 (yarı hassasiyet) kullanılarak GPU belleği optimize edilir
- 3.3M çift için olasılık üretilir, diske kaydedilir

**LightGBM submission:**
- TF-IDF ve kelime eşleşme featureları hesaplanır
- LightGBM ile olasılık üretilir, diske kaydedilir

In [4]:
# BERT Submission Olasılıkları
print("BERT submission olasılıkları hesaplanıyor...")

class SubDataset(Dataset):
    def __init__(self, queries, item_texts, tokenizer=None, max_length=64):
        self.queries = queries
        self.item_texts = item_texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.queries[idx], self.item_texts[idx],
            truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
        }

model.load_state_dict(torch.load(PATH + "best_model.pt"))
model = model.half()
model.eval()

sub_full = sub.merge(terms_clean, on="term_id", how="left")
sub_full = sub_full.merge(items_clean[["item_id","item_text"]], on="item_id", how="left")
sub_full["query"] = sub_full["query"].fillna("")
sub_full["item_text"] = sub_full["item_text"].fillna("")

sub_dataset = SubDataset(sub_full["query"].tolist(), sub_full["item_text"].tolist(), tokenizer=tokenizer)
sub_loader = DataLoader(sub_dataset, batch_size=512, shuffle=False, num_workers=0)

bert_probs = []
with torch.no_grad():
    for batch in tqdm(sub_loader, desc="BERT Predicting"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        bert_probs.extend(probs)

bert_probs = np.array(bert_probs)
np.save(PATH + "bert_probs.npy", bert_probs)
print(f"✅ BERT olasılıkları kaydedildi: {bert_probs.shape}")

BERT submission olasılıkları hesaplanıyor...


BERT Predicting: 100%|██████████| 6562/6562 [34:40<00:00,  3.15it/s] 

✅ BERT olasılıkları kaydedildi: (3359679,)


## LightGBM ile Submission Olasılıkları

**Featurelar:**
- TF-IDF skoru: query ile item_text arasındaki kelime ağırlıklı benzerlik
- title_score: query ile ürün başlığı arasındaki benzerlik
- category_score: query ile kategori arasındaki benzerlik
- jaccard: query ve item_text arasındaki kelime örtüşme oranı
- recall: query kelimelerinin item_text'te geçme oranı
- cat_jaccard/cat_recall: query ve kategori arasındaki örtüşme
- brand_match: query'de marka adı geçiyor mu
- query_exact_cat: query tam kategori metninde geçiyor mu
- query_exact_title: query tam başlık metninde geçiyor mu
- query_len/item_len: metin uzunlukları

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# Featurelar için orijinal items ve terms gerekiyor
items_orig = pd.read_csv(PATH + "items.csv")[["item_id","title","category","brand"]]
train_orig = pd.read_csv(PATH + "training_pairs.csv")
train_orig = train_orig.merge(terms_clean, on="term_id", how="left")
train_orig = train_orig.merge(items_clean[["item_id","item_text"]], on="item_id", how="left")
train_orig = train_orig.merge(items_orig, on="item_id", how="left")

pos_df_ml = train_orig[["query","item_text","title","category","brand"]].copy()
pos_df_ml["label"] = 1

neg_df_ml = dataset[dataset["label"]==0][["query","item_text"]].copy()
neg_df_ml["title"] = ""
neg_df_ml["category"] = ""
neg_df_ml["brand"] = ""
neg_df_ml["label"] = 0

dataset_ml = pd.concat([pos_df_ml, neg_df_ml], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
dataset_ml = dataset_ml.fillna("")

# Vektörizerlar
vec_full = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=50000, min_df=2)
vec_title = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=30000, min_df=2)
vec_cat = TfidfVectorizer(analyzer="word", ngram_range=(1,1), max_features=10000, min_df=2)

vec_full.fit(list(dataset_ml["query"]) + list(dataset_ml["item_text"]))
vec_title.fit(list(dataset_ml["query"]) + list(dataset_ml["title"]))
vec_cat.fit(list(dataset_ml["query"]) + list(dataset_ml["category"]))

def jaccard(q, t):
    q_set = set(str(q).lower().split())
    t_set = set(str(t).lower().split())
    if not q_set or not t_set:
        return 0.0
    return len(q_set & t_set) / len(q_set | t_set)

def query_recall(q, t):
    q_set = set(str(q).lower().split())
    t_set = set(str(t).lower().split())
    if not q_set:
        return 0.0
    return len(q_set & t_set) / len(q_set)

def brand_match(query, brand):
    if not query or not brand or brand == "unknown":
        return 0
    return 1 if str(brand).lower() in str(query).lower() else 0

def query_in_category(query, category):
    if not query or not category:
        return 0
    return 1 if str(query).lower() in str(category).lower() else 0

def query_in_title(query, title):
    if not query or not title:
        return 0
    return 1 if str(query).lower() in str(title).lower() else 0

def get_features(df):
    df = df.fillna("")
    q = vec_full.transform(df["query"])
    i = vec_full.transform(df["item_text"])
    tfidf = np.array((q.multiply(i)).sum(axis=1)).flatten()
    qt = vec_title.transform(df["query"])
    it = vec_title.transform(df["title"])
    title_sc = np.array((qt.multiply(it)).sum(axis=1)).flatten()
    qc = vec_cat.transform(df["query"])
    ic = vec_cat.transform(df["category"])
    cat_sc = np.array((qc.multiply(ic)).sum(axis=1)).flatten()
    return pd.DataFrame({
        "tfidf_score": tfidf,
        "title_score": title_sc,
        "category_score": cat_sc,
        "jaccard": df.apply(lambda r: jaccard(r["query"], r["item_text"]), axis=1).values,
        "recall": df.apply(lambda r: query_recall(r["query"], r["item_text"]), axis=1).values,
        "cat_jaccard": df.apply(lambda r: jaccard(r["query"], r["category"]), axis=1).values,
        "cat_recall": df.apply(lambda r: query_recall(r["query"], r["category"]), axis=1).values,
        "brand_match": df.apply(lambda r: brand_match(r["query"], r["brand"]), axis=1).values,
        "query_len": df["query"].str.split().str.len().values,
        "item_len": df["item_text"].str.split().str.len().values,
        "query_exact_cat": df.apply(lambda r: query_in_category(r["query"], r["category"]), axis=1).values,
        "query_exact_title": df.apply(lambda r: query_in_title(r["query"], r["title"]), axis=1).values,
    })

features = ["tfidf_score", "title_score", "category_score",
            "jaccard", "recall", "cat_jaccard", "cat_recall",
            "brand_match", "query_len", "item_len",
            "query_exact_cat", "query_exact_title"]

print("Train featureları hesaplanıyor...")
X = get_features(dataset_ml)
y = dataset_ml["label"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

model_lgb = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=127, min_child_samples=20, random_state=42, verbose=-1)
model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

from sklearn.metrics import f1_score
val_preds = model_lgb.predict(X_val)
print(f"✅ LightGBM Val F1: {f1_score(y_val, val_preds, average='macro'):.4f}")

# Submission olasılıkları
print("LightGBM submission featureları hesaplanıyor...")
sub_full2 = sub.merge(terms_clean, on="term_id", how="left")
sub_full2 = sub_full2.merge(items_clean[["item_id","item_text"]], on="item_id", how="left")
sub_full2 = sub_full2.merge(items_orig, on="item_id", how="left")
sub_full2 = sub_full2.fillna("")

X_sub = get_features(sub_full2)
ml_probs = model_lgb.predict_proba(X_sub)[:, 1]
np.save(PATH + "ml_probs.npy", ml_probs)
print(f"✅ LightGBM olasılıkları kaydedildi: {ml_probs.shape}")

Train featureları hesaplanıyor...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.065182
[200]	valid_0's binary_logloss: 0.0632418
Early stopping, best iteration is:
[181]	valid_0's binary_logloss: 0.0631899
✅ LightGBM Val F1: 0.9829
LightGBM submission featureları hesaplanıyor...
✅ LightGBM olasılıkları kaydedildi: (3359679,)



## Ensemble ve Final Submission

**Ensemble Yöntemi:**
BERT ve LightGBM modellerinin olasılıkları ağırlıklı ortalama ile birleştirildi.
- BERT ağırlığı: %80 (semantik anlama)
- LightGBM ağırlığı: %20 (kelime eşleşmesi)

**Threshold Optimizasyonu:**
Farklı eşik değerleri denenerek en yüksek Kaggle Public LB skorunu veren threshold seçildi.
- Threshold: 0.35 → Kaggle Public LB: **0.86**

**Not:** Public leaderboard test verisinin %15'i üzerinden hesaplanmaktadır. Final sıralama %85'lik private set üzerinden belirlenecektir.

In [6]:
# Olasılıkları yükle
bert_probs = np.load(PATH + "bert_probs.npy")
ml_probs = np.load(PATH + "ml_probs.npy")

# Ensemble: %80 BERT + %20 LightGBM
ensemble_probs = 0.8 * bert_probs + 0.2 * ml_probs

# Threshold 0.35 ile final submission
THRESHOLD = 0.35
final_preds = (ensemble_probs >= THRESHOLD).astype(int)

submission = pd.DataFrame({
    "id": sub["id"],
    "prediction": final_preds
})

submission.to_csv(PATH + "submission_final.csv", index=False)
print(f"✅ Final submission kaydedildi!")
print(f"1 (relevant)  : {final_preds.sum():,}")
print(f"0 (irrelevant): {(final_preds==0).sum():,}")
print(f"\nKaggle'a yüklenecek dosya: submission_final.csv")

✅ Final submission kaydedildi!
1 (relevant)  : 914,345
0 (irrelevant): 2,445,334

Kaggle'a yüklenecek dosya: submission_final.csv
